# P1 관세청 수입 축 — raw 정리·검증 + `clean_hsk_control` 적재

작성 2026-09-19 (담당분배 명세 `new_data/K_Defense_clean전환_담당분배_명세_20260918.md` §1·§2). 규칙 근거는 `docs/reference/data-cleaning-rules.md` §1·§2-1·§2-13·§2-15·§3·§4, `docs/reference/hs-whitelist-definition.md` §7·§8. DDL은 `db/alter_2026-09-19_p1_customs_hs.sql`(적용 2026-09-19, admin).

| # | 대상 | 하는 일 | 기대 |
|---|---|---|---|
| 1 | `raw_customs_trade` 294,420 → `fact_customs_monthly` 294,174 | **검증만**(재적재 없음). 총계행 제외·연월 분리·금액 단위·총계 대조 | 총계 246 제외, 불일치 0 |
| 2 | `raw_customs_progress` 264 | `row_count=0` 18건이 HS2022 신설 코드 공백인지 확인만 | 18 = 3 HS6 × 2016~2021 |
| 3 | `raw_hs_code_master` 12,469 | `dim_hs10` 현행 품목명·적용기간 열 UPDATE | 연결 107 / 미연결 104 |
| 4 | `raw_hs_unit_name` 17,072 | 코드 길이 실측 → 보정 규칙 기록, `ref_hs_rule_flag.hs6_name_ko` 채움 | 이름 열만 수정 |
| 5 | `raw_hsk_control` 2,161 | `clean_hsk_control` 신설 적재(1행 1통제번호) + `hsk_control_*` 지표 영향 대조 | ML 0건 = 「자료에 없음」 |
| 6 | `ref_hs_whitelist` 24 | CSV·문서 별표와 대조 — **고치지 않고 보고만** | 근거 미확인 5개 확인 |

원칙
- `raw_*`·`fact_customs_monthly`·`ref_hs_whitelist`·`ref_hs_rule_flag`의 **규칙 플래그 값**은 수정하지 않는다. 이 노트북이 쓰는 곳은 `dim_hs10`의 신규 열 4개, `ref_hs_rule_flag.hs6_name_ko`·`hs6_name_src`(이름 열), `clean_hsk_control`(비어 있을 때만), `meta_load_log`뿐이다.
- 관세청 수입액은 **민수 포함 국가 전체 수입**이다 — 「방산 수입」이라 쓰지 않는다.
- HS6 ↔ FSC 대응은 만들지 않는다(`ref_category_map` 미확정).
- 제외 행은 `clean_excluded_row`에 남기고 `raw = clean + excluded`를 검산한다. P1은 제외 0 기대(§5는 1:N 분해라 HSK10 고유 수로 검산).

In [1]:
import sys, re
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np

ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "scripts"))
from load_db import connect, log_stage, current_user   # .env 읽기 · meta_load_log 기록

NB_TAG = "notebooks/clean_p1_customs_hs.ipynb"
conn = connect()
cur = conn.cursor()
USER = current_user(cur)
NOW = datetime.now().replace(microsecond=0)
def q(sql, params=None):
    cur.execute(sql, params)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
def one(sql, params=None):
    cur.execute(sql, params); return cur.fetchone()
def count(t):
    cur.execute(f"SELECT COUNT(*) FROM {t}"); return cur.fetchone()[0]
def py(v):
    if v is None: return None
    if isinstance(v, float) and np.isnan(v): return None
    if v is pd.NaT or isinstance(v, pd._libs.missing.NAType): return None
    if hasattr(v, "item") and not isinstance(v, str): return v.item()
    if isinstance(v, pd.Timestamp): return v.to_pydatetime()
    return v
def insert_df(table, df, cols, chunk=2000):
    rows = [tuple(py(v) for v in r) for r in df[cols].itertuples(index=False, name=None)]
    sql = f"INSERT INTO {table} ({','.join(cols)}) VALUES ({','.join(['%s']*len(cols))})"
    for i in range(0, len(rows), chunk):
        cur.executemany(sql, rows[i:i+chunk])
    return len(rows)

print("connected as", USER, "| pandas", pd.__version__, "|", NOW)

connected as etl_rw@% | pandas 3.0.5 | 2026-09-19 07:30:53


## 1. `raw_customs_trade` → `fact_customs_monthly` 검증 (명세 §2 할 일 1, ★★★)

`fact_customs_monthly`는 이미 294,174행이라 **재적재하지 않고 검증만** 한다.

- **총계행 제외**: 원본의 연간 총계행은 `stat_ym='총계'`·`stat_cd='-'`·`hs_cd='-'`이며 `is_total='1'`로 표시된다. 한 호출(HS6 × 전 국가 × 연도)당 1행이라 246행(= 24 HS6 × 11연도 − 18. 빠진 18은 §2의 HS2022 신설 코드 공백). 상세행과 합치면 금액이 두 배가 되므로 fact에 넣지 않는다.
- **연·월 분리**: `stat_ym`(`YYYY.MM`) → `yyyymm`(점 제거 6자리) → `year`·`month`. 2026은 1~8월뿐이라 `is_partial_year=1`.
- **금액 단위**: 관세청 OpenAPI `getNitemtradeList`의 `expDlr`·`impDlr`이 **달러인지 천 달러인지**를 값의 자릿수로 판정한다(포털 설명 대신 실측). 판정 근거를 아래에 출력하고 `meta_column_dict`·`db/column_dict.csv` 설명에 반영했다(alter §5).
- **총계행 대조**: (`req_hs`, `req_year`)별 상세행 합 vs 총계행 값. 불일치 건수·HS·연도를 출력한다.

In [2]:
# 1-1 총계행 제외 · fact 1:1 대응
print("raw is_total 분포:", q("SELECT is_total, COUNT(*) n FROM raw_customs_trade GROUP BY is_total").set_index("is_total")["n"].to_dict())
print("총계행 형식 확인:", q("SELECT DISTINCT stat_ym, stat_cd, hs_cd FROM raw_customs_trade WHERE is_total='1'").to_dict("records"))
r_all, r_det, f_n = count("raw_customs_trade"), one("SELECT COUNT(*) FROM raw_customs_trade WHERE is_total='0'")[0], count("fact_customs_monthly")
print(f"원본 전체 {r_all:,} = 상세 {r_det:,} + 총계 {r_all-r_det:,} | fact {f_n:,} | 상세−fact = {r_det-f_n}")
print("상세행인데 fact 에 없음:", one("SELECT COUNT(*) FROM raw_customs_trade r LEFT JOIN fact_customs_monthly f ON f.raw_row_id=r.row_id WHERE r.is_total='0' AND f.raw_row_id IS NULL")[0])
print("fact 인데 상세행이 아님:", one("SELECT COUNT(*) FROM fact_customs_monthly f LEFT JOIN raw_customs_trade r ON r.row_id=f.raw_row_id WHERE r.row_id IS NULL OR r.is_total<>'0'")[0])

# 1-2 연·월 분리 확인 (raw stat_ym 'YYYY.MM' → fact yyyymm/year/month)
print("\n[연월 분리] 형식 위반 상세행:", one("SELECT COUNT(*) FROM raw_customs_trade WHERE is_total='0' AND stat_ym NOT REGEXP '^[0-9]{4}[.][0-9]{2}$'")[0])
print("fact yyyymm ≠ REPLACE(raw stat_ym,'.',''):",
      one("SELECT COUNT(*) FROM fact_customs_monthly f JOIN raw_customs_trade r ON r.row_id=f.raw_row_id WHERE f.yyyymm <> REPLACE(r.stat_ym,'.','')")[0])
print("year/month 파생 불일치:", one("SELECT COUNT(*) FROM fact_customs_monthly WHERE year<>CAST(LEFT(yyyymm,4) AS UNSIGNED) OR month<>CAST(RIGHT(yyyymm,2) AS UNSIGNED)")[0])
print("is_partial_year 규칙 위반(2026만 1이어야 함):", one("SELECT COUNT(*) FROM fact_customs_monthly WHERE (year=2026)<>(is_partial_year=1)")[0])
display(q("SELECT year, COUNT(*) rows_n, MIN(month) m0, MAX(month) m1, MAX(is_partial_year) partial FROM fact_customs_monthly GROUP BY year ORDER BY year"))

raw is_total 분포: {'0': 294174, '1': 246}


총계행 형식 확인: [{'stat_ym': '총계', 'stat_cd': '-', 'hs_cd': '-'}]


원본 전체 294,420 = 상세 294,174 + 총계 246 | fact 294,174 | 상세−fact = 0


상세행인데 fact 에 없음: 0


fact 인데 상세행이 아님: 0



[연월 분리] 형식 위반 상세행: 0


fact yyyymm ≠ REPLACE(raw stat_ym,'.',''): 0
year/month 파생 불일치: 0


is_partial_year 규칙 위반(2026만 1이어야 함): 0


,year,rows_n,m0,m1,partial
0,2016,26684,1,12,0
1,2017,29526,1,12,0
2,2018,29856,1,12,0
3,2019,29833,1,12,0
4,2020,28870,1,12,0
5,2021,29385,1,12,0
6,2022,25204,1,12,0
7,2023,25000,1,12,0
8,2024,25793,1,12,0
9,2025,26211,1,12,0


In [3]:
# 1-3 금액 단위 판정 — 포털 설명이 아니라 자릿수·단가로 본다
u = q("""SELECT hs6, SUM(imp_dlr) imp_2024, SUM(imp_wgt) kg_2024,
                 ROUND(SUM(imp_dlr)/NULLIF(SUM(imp_wgt),0),1) usd_per_kg
          FROM fact_customs_monthly WHERE year=2024 AND hs6 IN ('848620','854231','852610')
          GROUP BY hs6 ORDER BY hs6""")
display(u)
print("판정: 848620(반도체 제조용 기계) 2024 수입 {:,} — 달러로 읽으면 121억 달러(한국 2024 총수입 약 6,300억 달러의 1.9%)로 타당하고,".format(int(u.loc[u.hs6=='848620','imp_2024'].iloc[0])))
print("      천 달러로 읽으면 12조 달러라 한국 총수입의 약 19배가 된다. 854231 단가 {:,.0f} USD/kg 도 달러 기준에서만 성립한다(천 달러면 1천만 달러/kg).".format(float(u.loc[u.hs6=='854231','usd_per_kg'].iloc[0])))
print("      → impDlr·expDlr·balPayments 는 **달러 단위**. 열 설명에 반영(db/alter_2026-09-19_p1_customs_hs.sql §5, db/column_dict.csv).")
print("      주의: 이 금액은 국가 전체 수입·수출(민수 포함)이며 방산 수입이 아니다.")

,hs6,imp_2024,kg_2024,usd_per_kg
0,848620,12129034952,22799994,532.0
1,852610,390404367,1718368,227.2
2,854231,32273223739,3206978,10063.4


판정: 848620(반도체 제조용 기계) 2024 수입 12,129,034,952 — 달러로 읽으면 121억 달러(한국 2024 총수입 약 6,300억 달러의 1.9%)로 타당하고,
      천 달러로 읽으면 12조 달러라 한국 총수입의 약 19배가 된다. 854231 단가 10,063 USD/kg 도 달러 기준에서만 성립한다(천 달러면 1천만 달러/kg).
      → impDlr·expDlr·balPayments 는 **달러 단위**. 열 설명에 반영(db/alter_2026-09-19_p1_customs_hs.sql §5, db/column_dict.csv).
      주의: 이 금액은 국가 전체 수입·수출(민수 포함)이며 방산 수입이 아니다.


In [4]:
# 1-4 HS10 별 월 합계 = 총계행 대조 (호출 단위 = req_hs × req_year, 총계행은 HS10 을 구분하지 않는다)
gap = q("""SELECT t.req_hs, t.req_year, t.imp_tot, d.imp_sum, t.imp_tot-d.imp_sum imp_gap,
                  t.exp_tot, d.exp_sum, t.exp_tot-d.exp_sum exp_gap, t.bal_tot-d.bal_sum bal_gap
           FROM (SELECT req_hs, req_year, SUM(CAST(imp_dlr AS SIGNED)) imp_tot, SUM(CAST(exp_dlr AS SIGNED)) exp_tot,
                        SUM(CAST(bal_payments AS SIGNED)) bal_tot
                 FROM raw_customs_trade WHERE is_total='1' GROUP BY req_hs, req_year) t
           JOIN (SELECT req_hs, req_year, SUM(CAST(imp_dlr AS SIGNED)) imp_sum, SUM(CAST(exp_dlr AS SIGNED)) exp_sum,
                        SUM(CAST(bal_payments AS SIGNED)) bal_sum
                 FROM raw_customs_trade WHERE is_total='0' GROUP BY req_hs, req_year) d
             ON d.req_hs=t.req_hs AND d.req_year=t.req_year
           WHERE t.imp_tot<>d.imp_sum OR t.exp_tot<>d.exp_sum OR t.bal_tot<>d.bal_sum""")
n_tot = one("SELECT COUNT(*) FROM raw_customs_trade WHERE is_total='1'")[0]
print(f"총계행 {n_tot}개 대조 → 금액 불일치 {len(gap)}건")
if len(gap): display(gap)
else: print("  불일치 HS·연도 없음 (impDlr·expDlr·balPayments 전부 정확히 일치)")

# fact(상세행) 쪽에서도 같은 값이 나오는지 — 총계행은 fact 에 없으므로 fact 합 = 총계행 값이어야 한다
gap2 = q("""SELECT t.req_hs hs6, t.req_year yr, t.imp_tot, f.imp_f, t.imp_tot-f.imp_f gap
            FROM (SELECT req_hs, req_year, SUM(CAST(imp_dlr AS SIGNED)) imp_tot FROM raw_customs_trade WHERE is_total='1' GROUP BY 1,2) t
            JOIN (SELECT hs6, year, SUM(imp_dlr) imp_f FROM fact_customs_monthly GROUP BY 1,2) f
              ON f.hs6=t.req_hs AND f.year=t.req_year
            WHERE t.imp_tot<>f.imp_f""")
print("fact 합 vs 총계행 수입액 불일치:", len(gap2))
if len(gap2): display(gap2)
# 중량은 상세행이 kg 정수 반올림이라 오차가 있을 수 있다 — 참고값이므로 건수만 본다
wgap = q("""SELECT COUNT(*) n FROM (SELECT req_hs, req_year, SUM(CAST(imp_wgt AS SIGNED)) s FROM raw_customs_trade WHERE is_total='0' GROUP BY 1,2) d
            JOIN (SELECT req_hs, req_year, SUM(CAST(imp_wgt AS SIGNED)) s FROM raw_customs_trade WHERE is_total='1' GROUP BY 1,2) t
              ON t.req_hs=d.req_hs AND t.req_year=d.req_year WHERE t.s<>d.s""")
print("수입중량(kg) 불일치 호출 수:", int(wgap['n'].iloc[0]), "— 중량은 반올림 오차가 있는 참고값(data-sources.md:214)")

총계행 246개 대조 → 금액 불일치 0건
  불일치 HS·연도 없음 (impDlr·expDlr·balPayments 전부 정확히 일치)


fact 합 vs 총계행 수입액 불일치: 0


수입중량(kg) 불일치 호출 수: 227 — 중량은 반올림 오차가 있는 참고값(data-sources.md:214)


## 2. `raw_customs_progress` — `row_count=0` 18건 확인 (명세 §2 할 일 2)

수집 진행 기록(한 행 = HS6 × 국가범위 × 연도 호출). `row_count`는 API가 돌려준 행 수(총계행 포함)다. `row_count=0`이면 그 호출에 응답 자체가 없었다는 뜻이고, 그 HS6·연도는 `raw_customs_trade`·`fact_customs_monthly`에도 없어야 한다.

In [5]:
z = q("SELECT hs, year, row_count FROM raw_customs_progress WHERE row_count=0 ORDER BY hs, year")
print("progress 전체:", count("raw_customs_progress"), "| row_count=0:", len(z))
print("0인 HS6:", sorted(z['hs'].unique()), "| 연도:", sorted(z['year'].unique()))
# 이 HS6 는 화이트리스트에서 어떤 것인가 + HS2022 신설 여부(2022 이전 자료가 없다 = 신설 코드의 공백)
display(q("""SELECT p.hs, MIN(p.year) y_min0, MAX(p.year) y_max0, COUNT(*) zero_years,
                    (SELECT MIN(year) FROM raw_customs_progress p2 WHERE p2.hs=p.hs AND p2.row_count>0) first_data_year,
                    (SELECT name_ko FROM ref_hs_whitelist w WHERE w.hs6=p.hs) name_ko
             FROM raw_customs_progress p WHERE p.row_count=0 GROUP BY p.hs"""))
# progress 와 raw 실적의 정합
mism = q("""SELECT p.hs, p.year, p.row_count, COALESCE(r.n,0) raw_n
            FROM raw_customs_progress p
            LEFT JOIN (SELECT req_hs, req_year, COUNT(*) n FROM raw_customs_trade GROUP BY 1,2) r
              ON r.req_hs=p.hs AND r.req_year=p.year
            WHERE p.row_count <> COALESCE(r.n,0)""")
print("progress.row_count ≠ raw 적재 행 수:", len(mism))
if len(mism): display(mism)
f0 = q("""SELECT p.hs, p.year, COUNT(f.hs10) fact_rows FROM raw_customs_progress p
          LEFT JOIN fact_customs_monthly f ON f.hs6=p.hs AND f.year=p.year
          WHERE p.row_count=0 GROUP BY p.hs, p.year HAVING COUNT(f.hs10)>0""")
print("row_count=0 인데 fact 에 행이 있는 조합:", len(f0), "(0이어야 정합)")
print("판정: 854142·854159·880730 은 HS2022 신설 코드라 2016~2021 자료가 없다 → 18건(3 HS6 × 6년)은 수집 누락이 아니라 원자료 공백. fact 에서 빠진 것과 일치.")

progress 전체: 264 | row_count=0: 18
0인 HS6: ['854142', '854159', '880730'] | 연도: ['2016', '2017', '2018', '2019', '2020', '2021']


,hs,y_min0,y_max0,zero_years,first_data_year,name_ko
0,854142,2016,2021,6,2022,광전지(모듈 미조립)
1,854159,2016,2021,6,2022,기타 반도체 디바이스
2,880730,2016,2021,6,2022,항공기·헬리콥터 부분품


progress.row_count ≠ raw 적재 행 수: 0
row_count=0 인데 fact 에 행이 있는 조합: 0 (0이어야 정합)
판정: 854142·854159·880730 은 HS2022 신설 코드라 2016~2021 자료가 없다 → 18건(3 HS6 × 6년)은 수집 누락이 아니라 원자료 공백. fact 에서 빠진 것과 일치.


## 3. `raw_hs_code_master` → `dim_hs10` 보강 (명세 §2 할 일 3)

- `hs_desc`(HS부호내용)는 12,469행 **전부 빈 값**이라 옮기지 않는다(아래에서 재확인).
- 마스터는 7~10자리가 섞여 있다(7 52 / 8 918 / 9 172 / 10 11,327). `dim_hs10`은 HS10 표라 **10자리 11,327행만** 쓴다.
- `apply_start`·`apply_end`로 현행/이력을 가른다: 마스터 전 행의 `apply_end`가 2026-12-31이라 **마스터에 있으면 2026 현행 코드**, 없으면 과거 연도에만 있던 이력 코드다(관세청 응답에는 남아 있다). 값을 추정하지 않고 `master_link_status='마스터없음'` + NULL로 둔다.
- 기존 행은 지우지 않고 기존 `name_ko`(관세청 `statKor`)도 건드리지 않는다 — `v_hs10_use_tag_all`·`v_hs10_use_share`가 그 열을 읽는다.

In [6]:
print("hs_desc 빈 값:", one("SELECT SUM(hs_desc IS NULL OR hs_desc='') FROM raw_hs_code_master")[0], "/", count("raw_hs_code_master"), "→ 전부 비어 있으면 제외")
display(q("SELECT CHAR_LENGTH(hs_code) len, COUNT(*) n FROM raw_hs_code_master GROUP BY 1 ORDER BY 1"))
print("hs_code 중복:", one("SELECT COUNT(*) FROM (SELECT hs_code FROM raw_hs_code_master GROUP BY hs_code HAVING COUNT(*)>1) x")[0])
display(q("SELECT apply_end, COUNT(*) n FROM raw_hs_code_master GROUP BY 1"))

# UPDATE — 조인 키 dim_hs10.hs10 = raw_hs_code_master.hs_code (10자리)
cur.execute("""UPDATE dim_hs10 d JOIN raw_hs_code_master m ON m.hs_code = d.hs10 AND CHAR_LENGTH(m.hs_code)=10
               SET d.master_name_ko = m.name_ko,
                   d.apply_start = STR_TO_DATE(LEFT(m.apply_start,10), '%Y-%m-%d'),
                   d.apply_end   = STR_TO_DATE(LEFT(m.apply_end,10),   '%Y-%m-%d'),
                   d.master_link_status = '현행'""")
n_link = cur.rowcount
cur.execute("""UPDATE dim_hs10 d SET d.master_name_ko=NULL, d.apply_start=NULL, d.apply_end=NULL, d.master_link_status='마스터없음'
               WHERE NOT EXISTS (SELECT 1 FROM raw_hs_code_master m WHERE m.hs_code=d.hs10 AND CHAR_LENGTH(m.hs_code)=10)""")
conn.commit()
dim_n = count("dim_hs10")
linked = one("SELECT COUNT(*) FROM dim_hs10 WHERE master_link_status='현행'")[0]
print(f"\n[연결] dim_hs10 {dim_n}행 중 현행 마스터 연결 {linked} ({100*linked/dim_n:.1f}%) / 미연결(이력 코드) {dim_n-linked}")
display(q("""SELECT hs6, COUNT(*) hs10_n, SUM(master_link_status='현행') cur_n, SUM(master_link_status='마스터없음') hist_n
             FROM dim_hs10 GROUP BY hs6 HAVING hist_n>0 ORDER BY hist_n DESC"""))
display(q("SELECT apply_start, COUNT(*) n FROM dim_hs10 WHERE master_link_status='현행' GROUP BY 1 ORDER BY 1"))
print("\n품명 비교 표본(관세청 응답 statKor vs 마스터 한글품목명) — 다르면 마스터 쪽이 공식 분류표 표기:")
display(q("""SELECT hs10, LEFT(name_ko,40) name_ko_api, LEFT(master_name_ko,40) name_ko_master
             FROM dim_hs10 WHERE master_link_status='현행' AND name_ko <> master_name_ko LIMIT 8"""))
print("표기가 다른 HS10 수:", one("SELECT COUNT(*) FROM dim_hs10 WHERE master_link_status='현행' AND name_ko<>master_name_ko")[0])

hs_desc 빈 값: 12469 / 12469 → 전부 비어 있으면 제외


,len,n
0,7,52
1,8,918
2,9,172
3,10,11327


hs_code 중복: 0


,apply_end,n
0,2026-12-31 00:00:00,12469



[연결] dim_hs10 211행 중 현행 마스터 연결 107 (50.7%) / 미연결(이력 코드) 104


,hs6,hs10_n,cur_n,hist_n
0,851762,44,9,35
1,852990,28,10,18
2,848620,41,25,16
3,901380,12,3,9
4,854231,11,4,7
5,854233,11,4,7
6,854239,11,4,7
7,852910,7,4,3
8,847180,4,3,1
9,901480,3,2,1


,apply_start,n
0,2002-01-01,17
1,2007-01-01,11
2,2009-01-01,2
3,2012-01-01,1
4,2013-01-01,31
5,2014-01-01,2
6,2017-01-01,4
7,2022-01-01,31
8,2025-01-01,8



품명 비교 표본(관세청 응답 statKor vs 마스터 한글품목명) — 다르면 마스터 쪽이 공식 분류표 표기:


,hs10,name_ko_api,name_ko_master


표기가 다른 HS10 수: 0


## 4. `raw_hs_unit_name` 코드 길이 실측 → 보정 규칙 + `ref_hs_rule_flag.hs6_name_ko` (명세 §2 할 일 4)

명세는 「6시트 5자리, 8시트 7·9자리 = 앞자리 0 누락인지 확인」하라고 했다. **실측 결과 앞자리 0 누락이 아니다**:

1. 각 시트에 `0`으로 시작하는 코드가 이미 그대로 들어 있다(아래 출력). 0이 잘렸다면 01~09류 코드가 남아 있을 수 없다.
2. 5자리는 HS **한 줄(one-dash) 중간 수준**이다 — 예 `85423` = 「전자집적회로」이고 그 아래에 6자리 `854231`·`854232`·`854233`·`854239`가 따로 있다. 7·9자리도 8·10단위 사이의 중간 수준이다.
3. 대신 **자식이 하나뿐이라 세분되지 않는 HS6는 06시트에 행이 없다** — 예 `852692`(무선원격조종기기)는 10자리 `8526920000` 하나뿐이라 06시트에 없고 이름이 10시트에만 있다.

**보정 규칙(잠정, 이 노트북이 정함)**: HS6 공식 명칭 = ① 06시트 6자리 행 → ② 없으면 10시트에서 앞 6자리가 같은 행이 **정확히 1개**일 때 그 이름(`10시트단일`) → ③ 그래도 없으면 NULL(`없음`). 자리수를 늘리거나 0을 채워 넣지 않는다.

`ref_hs_rule_flag`는 **이름 열(`hs6_name_ko`·`hs6_name_src`)만** 갱신한다 — `r1~r4`·`is_candidate_provisional`·`priority_rule`·`evidence_rule` 등 판정 값은 건드리지 않는다.

In [7]:
display(q("SELECT hs_unit, CHAR_LENGTH(hs_code) len, COUNT(*) n, SUM(LEFT(hs_code,1)='0') lead_zero FROM raw_hs_unit_name GROUP BY 1,2 ORDER BY 1,2"))
print("→ 앞자리 0 이 살아 있는 행:", one("SELECT SUM(LEFT(hs_code,1)='0') FROM raw_hs_unit_name")[0], "행 → 0 누락 가설 기각")
print("\n5자리 코드 중 6자리 자식이 있는 것:",
      one("""SELECT COUNT(*) FROM (SELECT DISTINCT hs_code c FROM raw_hs_unit_name WHERE hs_unit='06' AND CHAR_LENGTH(hs_code)=5) a
              WHERE EXISTS (SELECT 1 FROM raw_hs_unit_name b WHERE b.hs_unit='06' AND CHAR_LENGTH(b.hs_code)=6 AND LEFT(b.hs_code,5)=a.c)""")[0],
      "/", one("SELECT COUNT(DISTINCT hs_code) FROM raw_hs_unit_name WHERE hs_unit='06' AND CHAR_LENGTH(hs_code)=5")[0])
display(q("SELECT hs_unit, hs_code, LEFT(name_ko,34) name_ko FROM raw_hs_unit_name WHERE hs_code IN ('85423','854231','854239','85269','852610','852691') ORDER BY hs_unit, hs_code"))
print("\n06시트에 6자리 행이 없는 화이트리스트 HS6:")
display(q("""SELECT w.hs6, w.name_ko whitelist_name,
                    (SELECT COUNT(*) FROM raw_hs_unit_name u WHERE u.hs_unit='10' AND LEFT(u.hs_code,6)=w.hs6) hs10_children,
                    (SELECT LEFT(u.name_ko,44) FROM raw_hs_unit_name u WHERE u.hs_unit='10' AND LEFT(u.hs_code,6)=w.hs6 LIMIT 1) name_from_10
             FROM ref_hs_whitelist w
             WHERE NOT EXISTS (SELECT 1 FROM raw_hs_unit_name u WHERE u.hs_unit='06' AND u.hs_code=w.hs6) ORDER BY w.hs6"""))

,hs_unit,len,n,lead_zero
0,02,2,97,9
1,04,4,1228,86
2,06,5,1024,106
3,06,6,2254,190
4,08,7,52,6
5,08,8,918,86
6,08,9,172,17
7,10,10,11327,1101


→ 앞자리 0 이 살아 있는 행: 1601 행 → 0 누락 가설 기각



5자리 코드 중 6자리 자식이 있는 것: 593 / 1024


,hs_unit,hs_code,name_ko
0,06,852610,레이더기기
1,06,85269,기타
2,06,852691,항행용 무선기기
3,06,85423,전자집적회로
4,06,854231,프로세서와 컨트롤러[메모리ㆍ변환기ㆍ논리회로ㆍ증폭기ㆍ클록(clo
5,06,854239,기타



06시트에 6자리 행이 없는 화이트리스트 HS6:


,hs6,whitelist_name,hs10_children,name_from_10
0,852692,무선원격조종기기,1,무선원격조절기기
1,854142,광전지(모듈 미조립),1,광전지(모듈에 조립되었거나 패널로 구성된 것은 제외한다)
2,901420,항공·우주용 항행기기,1,항공용이나 우주항행용 기기[컴퍼스(compass)는 제외한다]


In [8]:
RV = '2026-09-16'
before = q("SELECT COUNT(*) n, SUM(hs6_name_ko IS NOT NULL) has_name FROM ref_hs_rule_flag WHERE rule_version=%s", (RV,))
print("적용 전 ref_hs_rule_flag:", before.to_dict("records"))
flags_before = q("SELECT SUM(r1_mil) r1, SUM(r2_aero_nav) r2, SUM(r3_du) r3du, SUM(r3_ml) r3ml, SUM(r4_b2) r4, SUM(is_candidate_provisional) cand, SUM(in_whitelist) wl FROM ref_hs_rule_flag WHERE rule_version=%s", (RV,))

cur.execute("""UPDATE ref_hs_rule_flag f JOIN raw_hs_unit_name u ON u.hs_unit='06' AND u.hs_code=f.hs6
               SET f.hs6_name_ko=u.name_ko, f.hs6_name_src='06시트' WHERE f.rule_version=%s""", (RV,))
n06 = cur.rowcount
cur.execute("""UPDATE ref_hs_rule_flag f
               JOIN (SELECT LEFT(hs_code,6) hs6, MIN(name_ko) name_ko FROM raw_hs_unit_name WHERE hs_unit='10'
                     GROUP BY LEFT(hs_code,6) HAVING COUNT(*)=1) s ON s.hs6=f.hs6
               SET f.hs6_name_ko=s.name_ko, f.hs6_name_src='10시트단일'
               WHERE f.rule_version=%s AND f.hs6_name_src='없음'""", (RV,))
n10 = cur.rowcount
conn.commit()
print(f"이름 채움 — 06시트 {n06} · 10시트단일 {n10}")
display(q("SELECT hs6_name_src, COUNT(*) n, SUM(hs6_name_ko IS NOT NULL) has_name FROM ref_hs_rule_flag WHERE rule_version=%s GROUP BY 1", (RV,)))
flags_after = q("SELECT SUM(r1_mil) r1, SUM(r2_aero_nav) r2, SUM(r3_du) r3du, SUM(r3_ml) r3ml, SUM(r4_b2) r4, SUM(is_candidate_provisional) cand, SUM(in_whitelist) wl FROM ref_hs_rule_flag WHERE rule_version=%s", (RV,))
print("\n규칙 플래그 합계 — 전:", flags_before.to_dict("records")[0])
print("규칙 플래그 합계 — 후:", flags_after.to_dict("records")[0], "(같아야 한다: 이름 열만 수정)")
assert flags_before.equals(flags_after), "규칙 플래그가 바뀌었다 — 되돌려야 함"
print("\n화이트리스트 24개 공식 명칭:")
display(q("""SELECT f.hs6, f.hs6_name_src, LEFT(f.hs6_name_ko,46) hs6_name_ko FROM ref_hs_rule_flag f
             JOIN ref_hs_whitelist w ON w.hs6=f.hs6 WHERE f.rule_version=%s ORDER BY f.hs6""", (RV,)))

적용 전 ref_hs_rule_flag:

 [{'n': 1003, 'has_name': Decimal('998')}]


이름 채움 — 06시트 0 · 10시트단일 0


,hs6_name_src,n,has_name
0,10시트단일,489,489
1,06시트,509,509
2,없음,5,0



규칙 플래그 합계 — 전: {'r1': Decimal('3'), 'r2': Decimal('52'), 'r3du': Decimal('275'), 'r3ml': Decimal('0'), 'r4': Decimal('14'), 'cand': Decimal('58'), 'wl': Decimal('24')}
규칙 플래그 합계 — 후: {'r1': Decimal('3'), 'r2': Decimal('52'), 'r3du': Decimal('275'), 'r3ml': Decimal('0'), 'r4': Decimal('14'), 'cand': Decimal('58'), 'wl': Decimal('24')} (같아야 한다: 이름 열만 수정)

화이트리스트 24개 공식 명칭:


,hs6,hs6_name_src,hs6_name_ko
0,841191,06시트,터보제트나 터보프로펠러의 것
1,847180,06시트,그 밖의 자동자료처리기계의 단위 기기
2,848620,06시트,반도체 디바이스나 전자집적회로 제조용 기계와 기기
3,851762,06시트,음성ㆍ영상이나 그 밖의 자료의 수신용ㆍ변환용ㆍ송신용ㆍ재생용 기기[교환기와 라우팅(r
4,852560,06시트,수신기기를 갖춘 송신기기
5,852610,06시트,레이더기기
6,852691,06시트,항행용 무선기기
7,852692,10시트단일,무선원격조절기기
8,852910,06시트,"각종 안테나와 반사식 안테나, 그 부분품"
9,852990,06시트,기타


## 5. `raw_hsk_control` → `clean_hsk_control` 신설 (명세 §2 할 일 5)

- 원본 2,161행 = HSK10 2,161개(고유, 전 행 숫자 10자리). `control_no`는 쉼표 목록(최대 1,218자)이라 통제번호 단위 집계를 못 한다 → **1행 = HSK10 × 통제번호 1개**로 분해한다.
- 행이 늘어난 것은 **원본 규모가 아니다**(`CLAUDE.md` 데이터 검증 규칙). 원본 건수는 HSK10 2,161개, 이 표의 행 수는 「통제번호 부여 건수」로 따로 센다. 검산은 `raw HSK10 고유 = clean HSK10 고유 + 제외`.
- 통제번호 체계(대외무역법 §19 → 전략물자수출입고시): 첫 글자 = **부**(0~9), 둘째 글자 = 그룹(A~E). 3 전자 · 5 통신·정보보안 · 6 센서·레이저 · 7 항법·항공전자가 R3(전략물자-DU)가 쓰는 부다.
- **ML(별표3 군용물자)은 0건** — 「군용물자에 해당하는 HSK가 없다」가 아니라 **「이 자료(15034135)에 실려 있지 않다」**로 기록한다. 별표3 연계 자료는 국방기술품질원 소관이라 이 파일로는 확인할 수 없다.
- 이 표는 HS6 판정을 바꾸지 않는다 — `v_hs6_candidate_rule`·`v_hsk_control_by_hs6`는 `raw_hsk_control`을 그대로 읽는다.

In [9]:
PART_NAMES = {0:"원자력전용", 1:"특수재질", 2:"재료가공", 3:"전자", 4:"컴퓨터",
              5:"통신·정보보안", 6:"센서·레이저", 7:"항법·항공전자", 8:"해양", 9:"항공우주·추진"}
rk = q("SELECT row_id, hsk10, name_ko, control_no FROM raw_hsk_control ORDER BY row_id")
print("원본 전체:", len(rk), "| HSK10 고유:", rk["hsk10"].nunique(),
      "| 10자리 숫자 아님:", int((~rk["hsk10"].fillna("").str.fullmatch(r"[0-9]{10}")).sum()),
      "| control_no 공란:", int(rk["control_no"].fillna("").eq("").sum()))

recs = []
for row_id, hsk10, name_ko, ctrl in rk.itertuples(index=False, name=None):
    toks = [t.strip() for t in str(ctrl).split(",")]
    seq = 0
    for t in toks:
        if not t:
            continue
        seq += 1
        norm = t.upper().rstrip(".")
        is_ml = norm.startswith("ML")
        part = int(t[0]) if (not is_ml and t[0].isdigit()) else None
        grp = t[1].upper() if (not is_ml and len(t) > 1 and t[1].isalpha()) else None
        recs.append(dict(raw_row_id=row_id, hsk10=hsk10, hs6=hsk10[:6], hs2=hsk10[:2], name_ko=name_ko,
                         control_no=t, control_no_norm=norm,
                         regime="군용물자" if is_ml else "이중용도",
                         part_no=part, part_name_ko=PART_NAMES.get(part), group_code=grp,
                         is_du_elec=int(part in (3, 5, 6, 7)) if part is not None else 0,
                         seq_in_row=seq))
ck = pd.DataFrame(recs)
print("\n분해 토큰(빈 토큰 제외):", len(ck), "| 평균 통제번호/HSK10:", round(len(ck)/rk['hsk10'].nunique(), 2))
dup_mask = ck.duplicated(["hsk10", "control_no_norm"], keep="first")
print("같은 HSK10 안 통제번호 중복(정규화 후):", int(dup_mask.sum()), "→ 대표 1개만 남긴다(원본 행은 raw 에 그대로)")
ck = ck[~dup_mask].reset_index(drop=True)
ck["cleaned_at"] = NOW; ck["cleaned_by"] = USER
print("clean_hsk_control 적재 예정 행:", len(ck), "| HSK10 고유:", ck['hsk10'].nunique())
print("regime:", ck['regime'].value_counts().to_dict(), "← 군용물자(ML) 0 = 「없음」이 아니라 「자료에 없음」")
print("부 분포:", ck.groupby(['part_no'])['hsk10'].size().to_dict())
print("부 이름 없음(첫 글자가 숫자가 아님):", int(ck['part_no'].isna().sum()))
print("R3 부(3·5·6·7) 행:", int(ck['is_du_elec'].sum()), "| 그 HSK10 고유:", ck.loc[ck.is_du_elec==1,'hsk10'].nunique())
print("그룹:", ck['group_code'].value_counts(dropna=False).to_dict())

원본 전체: 2161 | HSK10 고유: 2161 | 10자리 숫자 아님: 0 | control_no 공란: 0

분해 토큰(빈 토큰 제외): 10133 | 평균 통제번호/HSK10: 4.69
같은 HSK10 안 통제번호 중복(정규화 후): 29 → 대표 1개만 남긴다(원본 행은 raw 에 그대로)
clean_hsk_control 적재 예정 행: 10104 | HSK10 고유: 2161
regime: {'이중용도': 10104} ← 군용물자(ML) 0 = 「없음」이 아니라 「자료에 없음」
부 분포: {0: 392, 1: 3052, 2: 1873, 3: 1287, 4: 263, 5: 574, 6: 811, 7: 488, 8: 209, 9: 1155}
부 이름 없음(첫 글자가 숫자가 아님): 0
R3 부(3·5·6·7) 행: 3160 | 그 HSK10 고유: 707
그룹: {'A': 3739, 'B': 2617, 'C': 2095, 'D': 836, 'E': 817}


In [10]:
if count("clean_hsk_control") > 0:
    print("clean_hsk_control 비어 있지 않음 → 적재 건너뜀 (재적재는 admin TRUNCATE 후 재실행)")
else:
    cols = ["raw_row_id","hsk10","hs6","hs2","name_ko","control_no","control_no_norm","regime",
            "part_no","part_name_ko","group_code","is_du_elec","seq_in_row","cleaned_at","cleaned_by"]
    n_ins = insert_df("clean_hsk_control", ck, cols)
    log_stage(cur, "kosti_hsk_control", "clean_hsk_control", "원본 전체", "HSK10 행", len(rk),
              "파서: DB raw_hsk_control(원본 CSV utf-8-sig 2,161행과 일치)", f"{NB_TAG} §5", USER)
    log_stage(cur, "kosti_hsk_control", "clean_hsk_control", "중복 처리 후", "HSK10 × 통제번호 1개(세로형)", n_ins,
              f"완전 중복 0 · 같은 HSK10 안 통제번호 중복 {int(dup_mask.sum())}건은 대표 1개만(원본 보존). "
              f"행 수 증가는 1:N 분해이며 원본 규모가 아니다 — 원본 건수는 HSK10 {rk['hsk10'].nunique()}개",
              f"{NB_TAG} §5", USER)
    log_stage(cur, "kosti_hsk_control", "clean_hsk_control", "관련 후보", "이중용도 3·5·6·7부(R3 전략물자-DU)",
              int(ck["is_du_elec"].sum()),
              f"해당 HSK10 고유 {ck.loc[ck.is_du_elec==1,'hsk10'].nunique()}개. ML(별표3 군용물자)은 이 자료에 0건 — 「자료에 없음」",
              f"{NB_TAG} §5", USER)
    log_stage(cur, "kosti_hsk_control", "clean_hsk_control", "관련 후보", "화이트리스트 24 HS6 아래 통제 HSK10",
              int(q("SELECT COUNT(DISTINCT c.hsk10) n FROM clean_hsk_control c JOIN ref_hs_whitelist w ON w.hs6=c.hs6")["n"].iloc[0])
              if False else int(ck.loc[ck["hs6"].isin(q("SELECT hs6 FROM ref_hs_whitelist")["hs6"]), "hsk10"].nunique()),
              "통제 = 수출허가 대상 「해당 가능성」이지 군용 확정이 아니다(hs-whitelist-definition.md §8-1 S2)",
              f"{NB_TAG} §5", USER)
    conn.commit()
    print("적재:", n_ins, "행 | meta_load_log 4행 기록")
display(q("SELECT part_no, part_name_ko, COUNT(*) rows_n, COUNT(DISTINCT hsk10) hsk_n FROM clean_hsk_control GROUP BY 1,2 ORDER BY 1"))
display(q("SELECT regime, COUNT(*) rows_n FROM clean_hsk_control GROUP BY 1"))

clean_hsk_control 비어 있지 않음 → 적재 건너뜀 (재적재는 admin TRUNCATE 후 재실행)


,part_no,part_name_ko,rows_n,hsk_n
0,0,원자력전용,392,191
1,1,특수재질,3052,1186
2,2,재료가공,1873,577
3,3,전자,1287,375
4,4,컴퓨터,263,70
5,5,통신·정보보안,574,205
6,6,센서·레이저,811,250
7,7,항법·항공전자,488,103
8,8,해양,209,107
9,9,항공우주·추진,1155,279


,regime,rows_n
0,이중용도,10104


In [11]:
# 5-2 지표·판정 영향 대조 — alter §4 에서 hsk_control_* 지표 48행을 넣었다.
#     (적용 전 실측 2026-09-19, DBHub app_ro: verdict 유지 19 / 신규 후보 39 / 강등·제외 5, civil_mix_basis hs10 9 / 판단불가 15)
display(q("SELECT indicator, COUNT(*) n, ROUND(MIN(value_num),2) mn, ROUND(AVG(value_num),2) avg_, ROUND(MAX(value_num),2) mx FROM ref_hs_indicator WHERE indicator LIKE 'hsk_control%' GROUP BY indicator"))
display(q("SELECT verdict, COUNT(*) n FROM v_hs6_candidate_vs_whitelist GROUP BY verdict ORDER BY verdict"))
print("→ 적용 후 verdict 분포가 위 「적용 전」과 같으면 HS6 24개 판정은 바뀌지 않은 것이다(화이트리스트 제외·추가 없음).")
display(q("SELECT civil_mix_basis, COUNT(*) n, SUM(civil_mix_rule IS NULL) label_null FROM v_civil_mix_rule GROUP BY 1"))
print("civil_mix 라벨(v_civil_mix_rule):", q("SELECT COALESCE(civil_mix_rule,'(NULL)') v, COUNT(*) n FROM v_civil_mix_rule GROUP BY 1")
      .set_index("v")["n"].to_dict())
print("※ 지표를 넣으면 basis 가 '판단불가'(15) → 'hsk'(15) 로 바뀌지만 문턱값이 미정이라 **라벨은 NULL 그대로**다.")
print("   저장 열 ref_hs_whitelist.civil_mix_basis 는 스냅샷이라 '판단불가' 로 남아 있다 — 뷰와 스냅샷 차이는 잠정(schema-design.md §7).")
display(q("""SELECT i.hs6, ROUND(r.value_num,1) ratio_pct, ROUND(s.value_num,1) imp_share_pct, w.civil_mix, w.civil_mix_basis
             FROM ref_hs_whitelist w
             JOIN ref_hs_indicator r ON r.hs6=w.hs6 AND r.indicator='hsk_control_hs10_ratio'
             JOIN ref_hs_indicator s ON s.hs6=w.hs6 AND s.indicator='hsk_control_imp_share'
             JOIN ref_hs_indicator i ON i.indicator_id=r.indicator_id ORDER BY i.hs6"""))

,indicator,n,mn,avg_,mx
0,hsk_control_hs10_ratio,24,0.00,63.33,100.00
1,hsk_control_imp_share,24,0.00,76.98,100.00


,verdict,n
0,강등·제외 검토(규칙 미해당),5
1,신규 후보(미수집),39
2,유지(근거 교체),19


→ 적용 후 verdict 분포가 위 「적용 전」과 같으면 HS6 24개 판정은 바뀌지 않은 것이다(화이트리스트 제외·추가 없음).


,civil_mix_basis,n,label_null
0,hsk,15,15
1,hs10,9,0


civil_mix 라벨(v_civil_mix_rule): {'(NULL)': 15, '높음': 4, '낮음': 2, '중간': 3}
※ 지표를 넣으면 basis 가 '판단불가'(15) → 'hsk'(15) 로 바뀌지만 문턱값이 미정이라 **라벨은 NULL 그대로**다.
   저장 열 ref_hs_whitelist.civil_mix_basis 는 스냅샷이라 '판단불가' 로 남아 있다 — 뷰와 스냅샷 차이는 잠정(schema-design.md §7).


,hs6,ratio_pct,imp_share_pct,civil_mix,civil_mix_basis
0,841191,100.0,100.0,낮음,hs10
1,847180,50.0,24.6,NaN,판단불가
2,848620,43.9,97.8,NaN,판단불가
3,851762,20.5,76.1,NaN,판단불가
4,852560,100.0,100.0,NaN,판단불가
5,852610,100.0,100.0,중간,hs10
6,852691,100.0,100.0,중간,hs10
7,852692,100.0,100.0,NaN,판단불가
8,852910,57.1,98.7,NaN,판단불가
9,852990,14.3,39.6,NaN,판단불가


## 6. `ref_hs_whitelist` 검수 (명세 §2 할 일 6) — **고치지 않고 보고만**

`data/reference/hs_whitelist.csv`(24행, 원본 열 이름은 `hs_code`)와 DB `ref_hs_whitelist`를 전 열 대조하고, `docs/reference/hs-whitelist-definition.md` §2 별표의 `evidence`·`civil_mix`·`priority`와도 맞춰 본다. 불일치는 값을 고치지 않고 아래 표로 남긴다(명세 §1-1 「ref 테이블은 검수」, 이번 과제 지시 「불일치는 고치지 말고 보고」).

In [12]:
wl_csv = pd.read_csv(ROOT / "data/reference/hs_whitelist.csv", dtype=str, keep_default_na=False, encoding="utf-8-sig")
wl_csv = wl_csv.rename(columns={"hs_code": "hs6"})
wl_db = q("SELECT * FROM ref_hs_whitelist")
print("CSV 행:", len(wl_csv), "| DB 행:", len(wl_db))
print("CSV 에만 있는 HS6:", sorted(set(wl_csv.hs6) - set(wl_db.hs6)), "| DB 에만:", sorted(set(wl_db.hs6) - set(wl_csv.hs6)))
cmp_cols = [c for c in wl_csv.columns if c in wl_db.columns and c != "hs6"]
EMPTY = {"", "None", "nan", "NaN", "<NA>", "NaT", "(빈값)"}
def norm(df):
    return df.apply(lambda col: ["(빈값)" if (v is None or str(v).strip() in EMPTY) else str(v).strip() for v in col])
a = norm(wl_csv.set_index("hs6")[cmp_cols]).sort_index()
b = norm(wl_db.set_index("hs6")[cmp_cols]).sort_index()
diff = [(h, c, a.loc[h, c], b.loc[h, c]) for h in a.index for c in cmp_cols if a.loc[h, c] != b.loc[h, c]]
print("\nCSV ↔ DB 불일치 셀:", len(diff), "(빈 문자열·NULL 은 둘 다 (빈값) 으로 정규화해 비교)")
if diff: display(pd.DataFrame(diff, columns=["hs6", "column", "csv", "db"]))
print("\n[분포 대조 — 문서 §2 별표 기준값과 비교]")
for col, expected in [("priority", {"1": 9, "2": 10, "3": 5}), ("evidence_basis", {"rule": 19, "팀판단": 5}),
                      ("category", {"반도체": 8, "전자부품": 13, "소재장비": 3}), ("axis", {"both": 12, "import": 9, "export": 3})]:
    got = b[col].value_counts().to_dict()
    print(f"  {col}: DB {got} | 문서 {expected} | {'일치' if got == expected else '불일치'}")
cm = wl_db["civil_mix"].fillna("(NULL)").value_counts().to_dict()
print(f"  civil_mix: DB {cm} | 문서 높음 4 / 중간 3 / 낮음 2 / NULL 15 | {'일치' if cm.get('높음')==4 and cm.get('중간')==3 and cm.get('낮음')==2 and cm.get('(NULL)')==15 else '불일치'}")
print("\n[근거 미확인 5개 표시 확인 — 847180·848620·851762·854142·854159]")
display(q("""SELECT hs6, priority, evidence_basis, evidence, COALESCE(civil_mix,'(NULL)') civil_mix, civil_mix_basis
             FROM ref_hs_whitelist WHERE hs6 IN ('847180','848620','851762','854142','854159') ORDER BY hs6"""))
bad = q("SELECT COUNT(*) n FROM ref_hs_whitelist WHERE hs6 IN ('847180','848620','851762','854142','854159') AND NOT (priority=3 AND evidence_basis='팀판단')")
print("규칙 미해당 5개 중 (priority=3 AND evidence_basis='팀판단') 위반:", int(bad['n'].iloc[0]))
print("rule 근거인데 priority 3 인 행:", int(q("SELECT COUNT(*) n FROM ref_hs_whitelist WHERE evidence_basis='rule' AND priority=3")['n'].iloc[0]))

CSV 행: 24 | DB 행: 24
CSV 에만 있는 HS6: [] | DB 에만: []

CSV ↔ DB 불일치 셀: 0 (빈 문자열·NULL 은 둘 다 (빈값) 으로 정규화해 비교)

[분포 대조 — 문서 §2 별표 기준값과 비교]
  priority: DB {'2': 10, '1': 9, '3': 5} | 문서 {'1': 9, '2': 10, '3': 5} | 일치
  evidence_basis: DB {'rule': 19, '팀판단': 5} | 문서 {'rule': 19, '팀판단': 5} | 일치
  category: DB {'전자부품': 13, '반도체': 8, '소재장비': 3} | 문서 {'반도체': 8, '전자부품': 13, '소재장비': 3} | 일치
  axis: DB {'both': 12, 'import': 9, 'export': 3} | 문서 {'both': 12, 'import': 9, 'export': 3} | 일치
  civil_mix: DB {'(NULL)': 15, '높음': 4, '중간': 3, '낮음': 2} | 문서 높음 4 / 중간 3 / 낮음 2 / NULL 15 | 일치

[근거 미확인 5개 표시 확인 — 847180·848620·851762·854142·854159]


,hs6,priority,evidence_basis,evidence,civil_mix,civil_mix_basis
0,847180,3,팀판단,팀판단,(NULL),판단불가
1,848620,3,팀판단,팀판단,(NULL),판단불가
2,851762,3,팀판단,팀판단,(NULL),판단불가
3,854142,3,팀판단,팀판단,(NULL),판단불가
4,854159,3,팀판단,A6;B2-FSC,(NULL),판단불가


규칙 미해당 5개 중 (priority=3 AND evidence_basis='팀판단') 위반: 0


rule 근거인데 priority 3 인 행: 0


## 7. `meta_load_log` 기록 + 최종 검산

검증 전용 항목(1·2·3·4·6)도 단계 건수를 남긴다. `clean_excluded_row`에 새로 넣을 행은 없다(P1 제외 0).

In [13]:
raw_all, raw_det, fact_n = count("raw_customs_trade"), one("SELECT COUNT(*) FROM raw_customs_trade WHERE is_total='0'")[0], count("fact_customs_monthly")
y2025 = one("SELECT COUNT(*) FROM fact_customs_monthly WHERE year=2025")[0]
combo = one("SELECT COUNT(*) FROM (SELECT DISTINCT hs6, stat_cd FROM fact_customs_monthly WHERE year=2025) x")[0]
already = one("SELECT COUNT(*) FROM meta_load_log WHERE dataset_key='customs_all' AND method LIKE %s", (f"{NB_TAG}%",))[0]
if already == 0:
    log_stage(cur, "customs_all", "raw_customs_trade", "원본 전체", "HS10 × 국가 × 월 + 연간 총계행", raw_all,
              "24 HS6 × 전 국가 × 2016.01~2026.08. 파서 기준(포털 표시 건수 아님)", f"{NB_TAG} §1", USER)
    log_stage(cur, "customs_all", "fact_customs_monthly", "중복 처리 후", "총계행 제외 상세행", fact_n,
              f"is_total='1' 연간 총계행 {raw_all-raw_det}행 제외(상세행과 합치면 금액 이중계상). 완전 중복 0 · 키 중복 0 · 총계 대조 불일치 0",
              f"{NB_TAG} §1", USER)
    log_stage(cur, "customs_all", "fact_customs_monthly", "선택 연도 원본", "2025 완결연도", y2025,
              f"2026 은 1~8월 부분연도(is_partial_year=1)라 연간 실적으로 쓰지 않는다. 2025 HS6×국가 조합 {combo}",
              f"{NB_TAG} §1", USER)
    log_stage(cur, "customs_progress", "raw_customs_progress", "원본 전체", "HS6 × 연도 호출 기록", count("raw_customs_progress"),
              "row_count=0 18건(854142·854159·880730 × 2016~2021)은 HS2022 신설 코드의 원자료 공백 — 수집 누락 아님. fact 에도 없음(확인)",
              f"{NB_TAG} §2", USER)
    log_stage(cur, "customs_hs_code_master", "dim_hs10", "관련 후보", "현행 마스터 연결 HS10",
              one("SELECT COUNT(*) FROM dim_hs10 WHERE master_link_status='현행'")[0],
              f"dim_hs10 {count('dim_hs10')}행 중. 미연결은 2026 마스터에 없는 이력 코드 — 값 추정 없이 NULL. 마스터 10자리 11,327행만 사용, hs_desc 는 전부 빈 값이라 제외",
              f"{NB_TAG} §3", USER)
    log_stage(cur, "customs_hs_unit_name", "ref_hs_rule_flag", "관련 후보", "HS6 공식 명칭 채운 행",
              one("SELECT COUNT(*) FROM ref_hs_rule_flag WHERE rule_version='2026-09-16' AND hs6_name_ko IS NOT NULL")[0],
              "06시트 6자리 → 없으면 10시트 단일 자식 이름. 5·7·9자리는 앞자리 0 누락이 아니라 HS 중간 수준(one-dash) — 0 채움 없음. 규칙 플래그 미변경",
              f"{NB_TAG} §4", USER)
    log_stage(cur, "ref_hs_whitelist", "ref_hs_whitelist", "검증된 분석 대상", "priority 1·2",
              one("SELECT COUNT(*) FROM ref_hs_whitelist WHERE priority IN (1,2)")[0],
              "근거 미확인 5개(847180·848620·851762·854142·854159)는 priority 3 배경. CSV ↔ DB 대조 결과는 §6",
              f"{NB_TAG} §6", USER)
    conn.commit()
    print("meta_load_log 기록 완료")
else:
    print("이미 이 노트북으로 기록된 meta_load_log 가 있어 건너뜀:", already, "행")
display(q("SELECT log_id, dataset_key, table_name, stage, stage_detail, row_count FROM meta_load_log WHERE method LIKE %s ORDER BY log_id", (f"{NB_TAG}%",)))

이미 이 노트북으로 기록된 meta_load_log 가 있어 건너뜀: 3 행


,log_id,dataset_key,table_name,stage,stage_detail,row_count
0,115,kosti_hsk_control,clean_hsk_control,원본 전체,HSK10 행,2161
1,116,kosti_hsk_control,clean_hsk_control,중복 처리 후,HSK10 × 통제번호 1개(세로형),10104
2,117,kosti_hsk_control,clean_hsk_control,관련 후보,이중용도 3·5·6·7부(R3 전략물자-DU),3160
3,118,kosti_hsk_control,clean_hsk_control,관련 후보,화이트리스트 24 HS6 아래 통제 HSK10,87
4,119,customs_all,raw_customs_trade,원본 전체,HS10 × 국가 × 월 + 연간 총계행,294420
5,120,customs_all,fact_customs_monthly,중복 처리 후,총계행 제외 상세행,294174
6,121,customs_all,fact_customs_monthly,선택 연도 원본,2025 완결연도,26211
7,122,customs_progress,raw_customs_progress,원본 전체,HS6 × 연도 호출 기록,264
8,123,customs_hs_code_master,dim_hs10,관련 후보,현행 마스터 연결 HS10,107
9,124,customs_hs_unit_name,ref_hs_rule_flag,관련 후보,HS6 공식 명칭 채운 행,998


In [14]:
# 최종 검산
display(q("""SELECT 'raw_hsk_control → clean_hsk_control' chk,
                    (SELECT COUNT(DISTINCT hsk10) FROM raw_hsk_control) raw_hsk10,
                    (SELECT COUNT(DISTINCT hsk10) FROM clean_hsk_control) clean_hsk10,
                    (SELECT COUNT(*) FROM clean_excluded_row WHERE table_name='raw_hsk_control') excluded,
                    (SELECT COUNT(*) FROM clean_hsk_control) clean_rows"""))
display(q("""SELECT 'raw_customs_trade → fact' chk, (SELECT COUNT(*) FROM raw_customs_trade) raw_n,
                    (SELECT COUNT(*) FROM raw_customs_trade WHERE is_total='1') total_rows,
                    (SELECT COUNT(*) FROM fact_customs_monthly) fact_n,
                    (SELECT COUNT(*) FROM raw_customs_trade) - (SELECT COUNT(*) FROM raw_customs_trade WHERE is_total='1')
                      - (SELECT COUNT(*) FROM fact_customs_monthly) gap"""))
display(q("SELECT master_link_status, COUNT(*) n FROM dim_hs10 GROUP BY 1"))
display(q("SELECT COUNT(*) meta_column_dict_rows FROM meta_column_dict"))
print("클린 표 키 중복:", one("SELECT COUNT(*) FROM (SELECT hsk10, control_no_norm FROM clean_hsk_control GROUP BY 1,2 HAVING COUNT(*)>1) x")[0])
conn.commit(); cur.close(); conn.close()
print("완료:", datetime.now().replace(microsecond=0))

,chk,raw_hsk10,clean_hsk10,excluded,clean_rows
0,raw_hsk_control → clean_hsk_control,2161,2161,0,10104


,chk,raw_n,total_rows,fact_n,gap
0,raw_customs_trade → fact,294420,246,294174,0


,master_link_status,n
0,현행,107
1,마스터없음,104


,meta_column_dict_rows
0,827


클린 표 키 중복: 0
완료: 2026-09-19 07:31:06
